# Pandas — GroupBy and Aggregation

> **Repo:** Python_Libraries | **Library:** Pandas  
> **Notebook:** 06_GroupBy_and_Aggregation  

---

# 06 - GroupBy and Aggregation in Pandas

GroupBy is one of the most powerful tools in Pandas for **split-apply-combine** workflows.

The idea:
1. **Split** — divide the DataFrame into groups based on some criteria
2. **Apply** — apply a function to each group independently
3. **Combine** — combine the results into a new DataFrame

Dataset used: `titanic.csv`

Why Titanic? - It has both categorical (Pclass, Sex) and numeric (Age, Fare) columns —  ideal for group-level analysis.

In [9]:
import pandas as pd
import numpy as np
import urllib.request
import os

# Download Titanic dataset if not already present
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
save_path = "datasets/titanic.csv"

os.makedirs("datasets", exist_ok=True)

if not os.path.exists(save_path):
    urllib.request.urlretrieve(url, save_path)
    print("Downloaded titanic.csv")
else:
    print("File already exists")

df = pd.read_csv(save_path)
print(df.shape)
df.head()

Downloaded titanic.csv
(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


##### Important Columns among the dataset

In [14]:
df[['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   Fare      891 non-null    float64
 5   Embarked  889 non-null    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 41.9+ KB


## Section 1: Groupby Basics

`groupby()` splits the DataFrame into groups based on one or more columns.

The result is a `DataFrameGroupBy` object — nothing is computed until we apply an aggregation.

**Syntax:**
```python
df.groupby('column')['target'].aggregation()
```

### 1.1 Single Column Groupby

In [22]:
# Average survival rate by Passenger Class
df.groupby('Pclass')['Survived'].mean()

Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64

#### Observation:

**1st Class** passengers had the highest survival rate among all the three classes -- reflects the class bias in Titanic evacuation process. 

### 1.2 Multiple Columns Groupby

In [27]:
# Average survival rate grouped by both Class and Sex
df.groupby(['Pclass', 'Sex'])['Survived'].mean()

Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

#### Observation:
**Females** were clearly given priority over **Males** in evacuation process in all the three classes.

## Section 2. Aggregation Functions

Once groups are formed, we apply aggregation functions to summarize each group.

| Function | Description |
|----------|-------------|
| `.sum()` | Total of values |
| `.mean()` | Average |
| `.count()` | Non-null count |
| `.min()` / `.max()` | Minimum / Maximum |
| `.std()` | Standard deviation |
| `.median()` | Median value |

### 2.1 Common Aggregations

In [33]:
# Fare statistics grouped by Passenger Class
print("Sum:\n", df.groupby('Pclass')['Fare'].sum())
print("\nMean:\n", df.groupby('Pclass')['Fare'].mean())
print("\nCount:\n", df.groupby('Pclass')['Fare'].count())
print("\nMin:\n", df.groupby('Pclass')['Fare'].min())
print("\nMax:\n", df.groupby('Pclass')['Fare'].max())
print("\nStd Dev:\n", df.groupby('Pclass')['Fare'].std())

Sum:
 Pclass
1    18177.4125
2     3801.8417
3     6714.6951
Name: Fare, dtype: float64

Mean:
 Pclass
1    84.154687
2    20.662183
3    13.675550
Name: Fare, dtype: float64

Count:
 Pclass
1    216
2    184
3    491
Name: Fare, dtype: int64

Min:
 Pclass
1    0.0
2    0.0
3    0.0
Name: Fare, dtype: float64

Max:
 Pclass
1    512.3292
2     73.5000
3     69.5500
Name: Fare, dtype: float64

Std Dev:
 Pclass
1    78.380373
2    13.417399
3    11.778142
Name: Fare, dtype: float64


## Section 3. `.agg()`: Multiple Aggregations at Once

Instead of running separate aggregations, `.agg()` lets you apply multiple functions in a single call — returns a clean summary DataFrame.

**Syntax:**
```python
df.groupby('column')['target'].agg(['func1', 'func2', ...])
```

In [40]:
# Multiple aggregations on Fare by Pclass in one shot
df.groupby('Pclass')['Fare'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])

,mean,median,std,min,max,count
Pclass,,,,,,
1,84.154687,60.2875,78.380373,0.0,512.3292,216
2,20.662183,14.2500,13.417399,0.0,73.5000,184
3,13.675550,8.0500,11.778142,0.0,69.5500,491


#### Observation

- 1st class fares are significantly higher and more spread out (high std dev)
- 3rd class fares are low and tightly clustered — less variation
- `.agg()` is far cleaner than running individual aggregations separately

### Section 4: Named Aggregations

Named aggregations let you **rename output columns** directly inside `.agg()`.

This produces a cleaner, more readable result — especially useful in reports and pipelines.

**Syntax:**
```python
df.groupby('column').agg(
    new_col_name=('source_column', 'aggregation_function')
)
```

In [47]:
# Clean summary with custom column names
df.groupby('Pclass').agg(
    avg_fare=('Fare', 'mean'),
    avg_age=('Age', 'mean'),
    survival_rate=('Survived', 'mean'),
    total_passengers=('PassengerId', 'count')
).round(2)

,avg_fare,avg_age,survival_rate,total_passengers
Pclass,,,,
1,84.15,38.23,0.63,216
2,20.66,29.88,0.47,184
3,13.68,25.14,0.24,491


#### Observation

- Named aggregations make output self-documenting — no need to rename columns afterward
- Multi-column aggregation in a single expression: Fare, Age, Survived, PassengerId all at once
- `round(2)` keeps the output clean

### Section 5. `transform()` — Group-level Values on Original Indexcolumns |

- `.transform()` applies a function per group but **returns a Series with the same index as the original DataFrame**
- so you can add group-level stats as new columns.

**Key difference from `.agg()`:**
| | `.agg()` | `.transform()` |
|---|---|---|
| Output shape | One row per group | Same shape as original df |
| Use case | Summary tables | Adding group stats as new columns |

In [57]:
# Add group mean fare as a new column (without merging)
df['class_mean_fare'] = df.groupby('Pclass')['Fare'].transform('mean')
df[['PassengerId', 'Pclass', 'Fare', 'class_mean_fare']].head(10)

,PassengerId,Pclass,Fare,class_mean_fare
0,1,3,7.2500,13.675550
1,2,1,71.2833,84.154687
2,3,3,7.9250,13.675550
3,4,1,53.1000,84.154687
4,5,3,8.0500,13.675550
5,6,3,8.4583,13.675550
6,7,1,51.8625,84.154687
7,8,3,21.0750,13.675550
8,9,3,11.1333,13.675550
9,10,2,30.0708,20.662183


### Section 6: `.filter()`: Filter Groups by a Condition

`.filter()` keeps or removes **entire groups** based on a condition applied to the group.

Individual rows are not filtered — the whole group passes or fails together.

**Syntax:**
```python
df.groupby('column').filter(lambda x: condition_on_group)
```

In [60]:
# Keeping only Pclass groups where average survival rate > 0.4
filtered_df = df.groupby('Pclass').filter(lambda x: x['Survived'].mean() > 0.4)

print("Original shape:", df.shape)
print("Filtered shape:", filtered_df.shape)
print("\nClasses retained:\n", filtered_df['Pclass'].unique())

Original shape: (891, 13)
Filtered shape: (400, 13)

Classes retained:
 [1 2]


#### Observation

- Only Pclass 1 (63%) and Pclass 2 (47%) pass the filter — Pclass 3 (24%) is excluded
- All rows belonging to Pclass 1 and 2 are retained, not just high-survival individuals
- Useful for removing low-sample or low-signal groups before modeling

### Section 7 `.apply()`: Custom Function per Group

`.apply()` is the most flexible GroupBy method.

It lets you pass any custom function that takes a group (sub-DataFrame) and returns a result.

**When to use `.apply()` over `.agg()`:**
- When your logic is too complex for a single aggregation
- When you need to operate on multiple columns together within a group
- When you want to return a transformed sub-DataFrame per group

In [89]:
# For each Pclass, get the top 3 passengers by Fare paid
def top_fare_passengers(group):
    return group.nlargest(3, 'Fare')[['Name', 'Fare', 'Survived']]

df.groupby('Pclass').apply(top_fare_passengers, include_groups=False)

Name      Fare  Survived
Pclass                                                            
1      258                    Ward, Miss. Anna  512.3292         1
       679  Cardeza, Mr. Thomas Drake Martinez  512.3292         1
       737              Lesurer, Mr. Gustave J  512.3292         1
2      72                 Hood, Mr. Ambrose Jr   73.5000         0
       120         Hickman, Mr. Stanley George   73.5000         0
       385           Davies, Mr. Charles Henry   73.5000         0
3      159          Sage, Master. Thomas Henry   69.5500         0
       180        Sage, Miss. Constance Gladys   69.5500         0
       201                 Sage, Mr. Frederick   69.5500         0

#### Observation:

- `.apply()` runs our custom function independently on each Pclass group
- `.apply()` is powerful but slower than `.agg()` — useful only when simpler methods aren't enough
- Top fare payers in 1st class paid 500+ and all survived
- Although top fare payers in other two classes did not survive

### Section 8: Aggregating Multiple Columns simultaneously 

`.agg()` can accept a **dictionary** to apply different functions to different columns
in a single call — very common in real-world EDA and reporting.

**Syntax:**
```python
df.groupby('column').agg({
    'col1': ['func1', 'func2'],
    'col2': 'func3'
})

In [108]:
# Different aggregations on different columns simultaneously
df.groupby('Pclass').agg({
    'Survived': ['sum', 'mean'],
    'Age':      ['median', 'std' ],
    'Fare':     ['max', 'mean']
}).round(2)

Survived          Age          Fare       
            sum  mean median   std     max   mean
Pclass                                           
1           136  0.63   37.0  14.8  512.33  84.15
2            87  0.47   29.0  14.0   73.50  20.66
3           119  0.24   24.0  12.5   69.55  13.68

#### Observation:

- Dictionary-style `.agg()` gives full control: each column gets its own set of functions
- Output has a MultiIndex column header — use `.columns` to flatten if needed
- This is the most common pattern in EDA reporting pipelines

### Section 9 Pivot Table: Groupby in Tabular Form

`pd.pivot_table()` is essentially a GroupBy presented as a 2D cross-tabulation.

It is easier to read when comparing two categorical dimensions simultaneously.

**Syntax:**
```python
pd.pivot_table(df, values='target', index='row_group', 
               columns='col_group', aggfunc='mean')
```

In [112]:
# Survival rate by Pclass (rows) and Sex (columns)
pd.pivot_table(
    df,
    values='Survived',
    index='Pclass',
    columns='Sex',
    aggfunc='mean'
).round(2)

Sex,female,male
Pclass,,
1,0.97,0.37
2,0.92,0.16
3,0.50,0.14


#### Observation:

- Pivot table gives the same result as `groupby(['Pclass','Sex'])['Survived'].mean()`
- But the 2D layout makes cross-comparison much easier to read
- Preferred format for presenting results to stakeholders

In [117]:
# Getting same result without `.pivot_table()`:
df.groupby(['Pclass', 'Sex'])['Survived'].mean()

Pclass  Sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
3       female    0.500000
        male      0.135447
Name: Survived, dtype: float64

### Section 10. GroupBy Quick Reference

| Method | Returns | Best For |
|--------|---------|----------|
| `.agg(['mean','sum'])` | Reduced DataFrame | Standard summaries |
| `.agg(name=('col','func'))` | Renamed columns | Clean reporting |
| `.agg({'col1':'mean','col2':'sum'})` | Multi-column summary | EDA pipelines |
| `.transform('mean')` | Same shape as df | Adding group stats as new column |
| `.filter(lambda x: ...)` | Subset of df | Removing low-signal groups |
| `.apply(custom_func)` | Flexible | Complex custom logic per group |
| `pd.pivot_table()` | 2D cross-tab | Stakeholder-ready summaries |